# Differential Expression Analysis

## TCGA-BRCA Molecular Subtypes

This notebook performs differential gene expression analysis between transcriptomics-derived breast cancer subtypes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind

In [ ]:
#Load Expression Matrix
expr = pd.read_csv(
    "/home/sonia/bioinformatics/cancer-subtyping-multiomics/final_data/normalized_expression.csv",
    index_col=0
)

expr.head()

In [ ]:
#Load Cluster Labels
clusters = pd.read_csv(
    "/home/sonia/bioinformatics/cancer-subtyping-multiomics/results/pca_clusters.csv",
    index_col=0
)

clusters.head()

In [ ]:
#Assign Subtype Labels
label_map = {
    0: "Basal-like",
    1: "Luminal B",
    2: "Luminal A"
}

clusters["Subtype"] = clusters["Cluster"].map(label_map)

clusters.head()

In [ ]:
#Match Samples
common_samples = expr.columns.intersection(clusters.index)

expr = expr[common_samples]

clusters = clusters.loc[common_samples]

print(expr.shape)
print(clusters.shape)

In [ ]:
#Compare Basal-like vs Luminal A
group1 = clusters[clusters["Subtype"] == "Basal-like"].index
group2 = clusters[clusters["Subtype"] == "Luminal A"].index

In [ ]:
#DEG Analysis
results = []

for gene in expr.index:

    vals1 = expr.loc[gene, group1]
    vals2 = expr.loc[gene, group2]

    stat, pval = ttest_ind(vals1, vals2)

    logfc = vals1.mean() - vals2.mean()

    results.append([gene, logfc, pval])

deg = pd.DataFrame(
    results,
    columns=["Gene", "logFC", "pvalue"]
)

deg.head()

In [ ]:
#Adjust P-values
from statsmodels.stats.multitest import multipletests

deg["adj_pvalue"] = multipletests(
    deg["pvalue"],
    method="fdr_bh"
)[1]

deg.head()

In [ ]:
#Significant Genes
sig = deg[
    (deg["adj_pvalue"] < 0.05) &
    (abs(deg["logFC"]) > 1)
]

print("Significant genes:", sig.shape[0])

sig.head()

In [ ]:
#Top Differentially Expressed Genes
sig.sort_values("adj_pvalue").head(20)

In [ ]:
#Volcano Plot
deg["minus_log10_p"] = -np.log10(deg["adj_pvalue"])

plt.figure(figsize=(8,6))

plt.scatter(
    deg["logFC"],
    deg["minus_log10_p"],
    s=10
)

plt.xlabel("log Fold Change")
plt.ylabel("-log10 adjusted p-value")
plt.title("Volcano Plot")

plt.show()

In [ ]:
#Top Genes Heatmap
top_genes = sig.sort_values("adj_pvalue").head(20)["Gene"]

heatmap_data = expr.loc[top_genes]

plt.figure(figsize=(10,8))

sns.heatmap(heatmap_data)

plt.title("Top Differentially Expressed Genes")

plt.show()

In [ ]:
#Save DEG Results
deg.to_csv(
    "/home/sonia/bioinformatics/cancer-subtyping-multiomics/results/differential_expression_results.csv",
    index=False
)

In [ ]:
# Interpretation

Differential expression analysis identified subtype-associated transcriptomic signatures between basal-like and luminal tumors.

Basal-like tumors demonstrated increased expression of proliferation-associated and EGFR-related genes, whereas luminal tumors showed enrichment of hormone receptor-associated genes.

These findings support biologically meaningful subtype separation and potential biomarker discovery.